# BERT Masked Language Modeling 예제

Kaggle Notebook 환경에서 Hugging Face `transformers`를 사용해 `bert-base-uncased` 모델로 Masked Language Modeling 추론을 수행합니다.

이 노트북은 다음 흐름으로 구성됩니다.

1. 필요한 라이브러리 설치 및 불러오기
2. `AutoTokenizer`와 `AutoModelForMaskedLM` 로드
3. `[MASK]` 토큰이 포함된 문장 입력
4. PyTorch 기반 추론 수행
5. `[MASK]` 위치의 top-5 예측 단어 출력

In [ ]:
# Kaggle Notebook에서 transformers가 없는 경우를 대비해 설치합니다.
# 이미 설치되어 있다면 대부분 빠르게 넘어갑니다.
%pip install -q transformers

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

In [ ]:
# GPU가 있으면 GPU를 사용하고, 없으면 CPU를 사용합니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 장치: {device}")

## 1. Tokenizer와 Masked LM 모델 불러오기

In [ ]:
model_name = "bert-base-uncased"

# AutoTokenizer는 bert-base-uncased에 맞는 토크나이저 설정을 자동으로 불러옵니다.
tokenizer = AutoTokenizer.from_pretrained(model_name)

# AutoModelForMaskedLM은 [MASK] 위치의 단어를 예측하는 모델 헤드를 포함합니다.
model = AutoModelForMaskedLM.from_pretrained(model_name)
model.to(device)
model.eval()

print(f"모델 로드 완료: {model_name}")
print(f"마스크 토큰: {tokenizer.mask_token}")
print(f"마스크 토큰 ID: {tokenizer.mask_token_id}")

## 2. [MASK] 토큰이 포함된 문장 입력

In [ ]:
# BERT의 마스크 토큰은 대문자 [MASK] 형식을 사용합니다.
text = "The capital of France is [MASK]."

print(text)

## 3. 문장 토크나이징 및 [MASK] 위치 찾기

In [ ]:
# 입력 문장을 PyTorch 텐서로 변환합니다.
inputs = tokenizer(text, return_tensors="pt")

# 모델과 같은 장치로 입력 텐서를 이동합니다.
inputs = {key: value.to(device) for key, value in inputs.items()}

# input_ids에서 [MASK] 토큰의 위치를 찾습니다.
mask_token_indices = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]

# 이 예제는 하나의 [MASK] 토큰을 사용하는 흐름입니다.
if len(mask_token_indices) != 1:
    raise ValueError(f"이 예제는 [MASK] 토큰 1개를 기대합니다. 현재 개수: {len(mask_token_indices)}")

mask_token_index = mask_token_indices.item()

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print("토큰화 결과:")
print(tokens)
print(f"[MASK] 위치: {mask_token_index}")

## 4. PyTorch로 추론 수행

In [ ]:
# torch.no_grad()는 추론 시 gradient 계산을 비활성화해 메모리 사용량을 줄입니다.
with torch.no_grad():
    outputs = model(**inputs)

# logits shape: (batch_size, sequence_length, vocab_size)
logits = outputs.logits

# [MASK] 위치에 해당하는 vocabulary 전체의 점수만 가져옵니다.
mask_token_logits = logits[0, mask_token_index, :]

# softmax를 적용해 vocabulary별 확률로 변환합니다.
mask_token_probabilities = torch.softmax(mask_token_logits, dim=-1)

print(f"logits shape: {tuple(logits.shape)}")
print(f"vocab size: {mask_token_probabilities.shape[0]}")

## 5. [MASK] 위치의 Top-5 예측 단어 출력

In [ ]:
# 확률이 높은 순서대로 top-5 토큰을 가져옵니다.
top_k = 5
top_probabilities, top_token_ids = torch.topk(mask_token_probabilities, k=top_k)

print("Top-5 예측 결과")
print("-" * 80)
print(f"{'rank':>4} | {'token_id':>8} | {'probability':>12} | {'token':<12} | completed sentence")
print("-" * 80)

for rank, (token_id_tensor, probability_tensor) in enumerate(zip(top_token_ids, top_probabilities), start=1):
    token_id = token_id_tensor.item()
    token = tokenizer.decode([token_id]).strip()
    probability = probability_tensor.item()
    completed_text = text.replace(tokenizer.mask_token, token)
    print(f"{rank:>4} | {token_id:>8} | {probability:>11.4%} | {token:<12} | {completed_text}")